# 10. Production — testing, deployment, observability

## Learning Objectives

LangGraph Learn how to test, deploy, and monitor your apps.

## 10.1 Environment Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4.1")

In [ ]:
# Observability settings (optional) - LangSmith or Langfuse
# Set the key in .env, or uncomment it below and enter it yourself.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: Automatically activated when LANGSMITH_TRACING=true (no code modification required)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: Pass config={"callbacks": [langfuse_handler]} when calling invoke/stream
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


## 10.2 App structure — langgraph.json

- `langgraph.json`: Graph definition, dependencies, Environment Variables settings
- `langgraph dev`: Run local development server

In [ ]:
import json

config = {
    "dependencies": ["."],
    "graphs": {
        "agent": "./agent.py:graph"
    },
    "env": ".env"
}
print("langgraph.json example:")
print(json.dumps(config, indent=2))
print()
print("Command:")
print("  $ pip install 'langgraph-cli[inmem]'")
print("$ langgraph dev # http://localhost:2024에서 Start local server")

## 10.3 LangGraph Studio — Visual Debugging tool

Studio is automatically provided when you run `langgraph dev`.

**Function:**
- Graph structure visualization
- Real-time execution tracking
- Check and modify state
- Interactive testing
- Checkpoint navigation (time travel)

**How to use:**```bash
$ langgraph dev
# 브라우저에서 http://localhost:2024 접속
# 또는 LangSmith Studio에서 원격 접속
```

## 10.4 Agent Chat UI

Talk to agent using the chat interface:```bash
$ npx @anthropic-ai/agent-chat-ui
```
**Function:**
- Real-time streaming chat
- tool calling Visualization
- Conversation branching
- Human-in-the-loop approved
- multi-agent Message classification

## 10.5 Test — Deterministic agent test

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# Graph to test
class TestState(TypedDict):
    input: str
    output: str


def process(state: TestState) -> dict:
    return {"output": state["input"].upper()}


builder = StateGraph(TestState)

builder.add_node("process", process)
builder.add_edge(START, "process")
builder.add_edge("process", END)

graph = builder.compile()


# Unit tests
def test_process():
    result = graph.invoke({"input": "hello"})

    assert result["output"] == "HELLO", f"HELLO 예상, {result['output']} 반환됨"

    print("  OK test_process")


def test_empty_input():
    result = graph.invoke({"input": ""})

    assert result["output"] == "", f"빈 문자열 예상, {result['output']} 반환됨"

    print("  OK test_empty_input")


print("Running tests:")

test_process()
test_empty_input()

print("All tests passed!")

## 10.6 LLM agent Test — Using GenericFakeChatModel

In [ ]:
from langchain_core.language_models import GenericFakeChatModel
from langchain.messages import AIMessage, HumanMessage, AnyMessage
from langgraph.graph import StateGraph, START, END, MessagesState

# Deterministic fake model
fake_model = GenericFakeChatModel(
    messages=iter(
        [
            AIMessage(content="The answer is 42."),
        ]
    )
)

def chatbot(state: MessagesState) -> dict:
    return {
        "messages": [fake_model.invoke(state["messages"])]
    }

builder = StateGraph(MessagesState)

builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

test_graph = builder.compile()

result = test_graph.invoke(
    {
        "messages": [HumanMessage(content="test")]
    }
)

assert "42" in result["messages"][-1].content

print("GenericFakeChatModel test passed!")
print(f"  응답: {result['messages'][-1].content}")

## 10.7 Deployment Options

**1. LangGraph Platform (managed):**
```bash
$ langgraph deploy
```

**2. Self-hosted Docker:**
```bash
$ langgraph build -t my-agent
$ docker run -p 2024:2024 my-agent
```
**3. LangGraph Cloud:**
- Automatic distribution linked to GitHub
- Managed by https://smith.langchain.com

## 10.8 observability — LangSmith Tracing

**Settings (`.env`):**```
LANGSMITH_API_KEY=lsv2-...
LANGSMITH_TRACING=true
```
**Automatically tracked items:**
- Each node execution time
- LLM input/output, token usage
- tool calling and results
- state Change
- Errors and retries

In [ ]:
import os

tracing = os.environ.get("LANGSMITH_TRACING", "false")
api_key = os.environ.get("LANGSMITH_API_KEY", "")

print("Current state:")

print(f"  트레이싱: {'활성화' if tracing == 'true' else '비활성화'}")
print(f"  API 키: {'설정됨' if api_key else '미설정'}")

## 10.9 Pregel Runtime Overview

- **Pregel** is the internal execution engine of LangGraph
- Both Graph API and Functional API run on Pregel
- Key concepts: **superstep**, **Channel**, **Checkpoint**
- **superstep**: Unit in which nodes of the same level are executed in parallel
- Generally no need to use it directly (Graph/Functional API abstracts it)

**LangGraph Execution Model:**```
[Super-step 1] Node A, Node B (병렬)
     ↓ 상태 업데이트
[Super-step 2] Node C (A, B 결과 기반)
     ↓ 상태 업데이트
[Super-step 3] Node D
     ↓
END
```
**Each superstep:**
1. Parallel execution of relevant nodes
2. Update state (apply reducer)
3. Save checkpoint
4. Next superstep decision

## 10.10 Production Checklist

| Item | tool | Description |
|------|------|------|
| unit testing | pytest | Test individual node functions |
| Integration Testing | GenericFakeChatModel | Full flow without API calls |
| persistence | PostgreSaver | Production checkpointer |
| observability | LangSmith | Tracing, Monitoring |
| Distribution | langgraph deploy | Managed Deployment |
| UI | Agent Chat UI | User Interface |

## 10.11 Summary

| Topic | Key Concepts |
|------|----------|
| App Structure | Set up project with `langgraph.json` |
| Studio | Visual debugging with `langgraph dev` |
| test | Deterministic Testing + GenericFakeChatModel |
| Distribution | Platform, Docker, Cloud options |
| observability | LangSmith Tracing |
| runtime | Pregel superstep Execution Model → Deeper in [13번 __TERM_104__북](13_api_guide_and_pregel.ipynb) |

### Next Steps
→ Proceed to **[11. Local Server](11_local_server.ipynb)**!
→ Skip to **[__TERM_007__ __TERM_100__ 과정](../04_deepagents/01_introduction.ipynb)**